# ExoPy local smoke test

This notebook exercises the current project locally, then finishes with a narrow public DACE download of one `S1D_A` FITS file through `Star.query_observations()`.

In [1]:
from pathlib import Path

import numpy as np
from astropy.io import fits

from exopy import Data, Instrument, Observation, Star
from exopy.storage import HDF5Store

workdir = Path("notebook_scratch")
workdir.mkdir(exist_ok=True)

print("Imports OK")

Imports OK


## Data wrapper

Try the basic transformation helpers on arrays.

In [2]:
data = Data(
    arrays={
        "time": np.array([1.0, 2.0, 3.0, 4.0]),
        "rv": np.array([10.0, np.nan, 12.0, 13.0]),
        "flux": np.array([100.0, 101.0, 99.0, 98.0]),
    },
    metadata={"target": "TOI178"},
)

clean = data.mask_invalid("rv")
normalized = clean.normalize("flux")

normalized.to_pandas()

,time,rv,flux
0,1.0,10.0,1.224745
1,3.0,12.0,0.000000
2,4.0,13.0,-1.224745


## Instrument filters

Build the filter payload that will be sent to DACE.

In [3]:
instrument = Instrument(name="ESPRESSO", version="19", mode="HR", drs_version="latest")

instrument.dace_name, instrument.as_filters()

('ESPRESSO19',
 {'instrument_name': {'equals': ['ESPRESSO19']},
  'instrument_mode': {'equals': ['HR']}})

## Star with a fake DACE client

This checks the orchestration logic without making a network call.

In [4]:
class FakeDaceClient:
    def get_star_properties(self, target):
        return {"target_name": target, "teff": 5700, "source": "fake"}

    def query_observations(self, filters, file_type=None, drs_version=None, limit=None):
        rows = [
            {"spectrum_id": 1, "product": "TOI178_S1D_A.fits", "file_type": "S1D_A"},
            {"spectrum_id": 2, "product": "TOI178_S1D_B.fits", "file_type": "S1D_B"},
        ]
        return rows[:limit]


star = Star("TOI178", cache_dir=workdir / "cache", client=FakeDaceClient())

properties = star.fetch_properties()
observations = star.query_observations(instrument=instrument, file_type="S1D_A", limit=5)

properties, observations, star.products, star.available_instruments(), star.available_file_types(), star.observation_date_range()

({'target_name': 'TOI178', 'teff': 5700, 'source': 'fake'},
 [{'spectrum_id': 1, 'product': 'TOI178_S1D_A.fits', 'file_type': 'S1D_A'},
  {'spectrum_id': 2, 'product': 'TOI178_S1D_B.fits', 'file_type': 'S1D_B'}],
 [],
 ['S1D_A', 'S1D_B'],
 (None, None))

## FITS to Observation

Create a tiny local FITS file, then load it through `Observation.from_fits()`.

In [5]:
fits_path = workdir / "toy_observation.fits"

primary = fits.PrimaryHDU()
primary.header["OBJECT"] = "TOI178"
primary.header["INSTRUME"] = "ESPRESSO19"
primary.header["DRS_VER"] = "latest"
primary.header["SPECT_ID"] = "toy-1"

flux_hdu = fits.ImageHDU(data=np.array([1.0, 2.0, 3.0]), name="FLUX")
fits.HDUList([primary, flux_hdu]).writeto(fits_path, overwrite=True)

observation = Observation.from_fits(fits_path)

{
    "target": observation.target_name,
    "instrument": observation.instrument_name,
    "arrays": list(observation.require_data().arrays),
    "flux": observation.require_data().arrays["flux"].tolist(),
}

{'target': 'TOI178',
 'instrument': 'ESPRESSO19',
 'arrays': ['flux'],
 'flux': [1.0, 2.0, 3.0]}

## HDF5 round trip

Persist the observation and load it back from the local HDF5 store.

In [6]:
store = HDF5Store(workdir / "observations.h5")
store.write_observation(observation)

loaded = store.read_observations("TOI178")
loaded_observation = loaded[0]

{
    "count": len(loaded),
    "spectrum_id": loaded_observation.metadata.spectrum_id,
    "flux": loaded_observation.require_data().arrays["flux"].tolist(),
}

{'count': 1, 'spectrum_id': 'toy-1', 'flux': [1.0, 2.0, 3.0]}

## Real public DACE download

This browses public products and downloads a single `S1D_A` FITS file through the `Star` object.

In [7]:
download_dir = workdir / "real_dace_download"
download_dir.mkdir(exist_ok=True)

live_star = Star("TOI178", cache_dir=download_dir / "cache")
live_instrument = Instrument(name="ESPRESSO", version="19", drs_version="latest")

live_observations = live_star.query_observations(
    instrument=live_instrument,
    file_type="S1D_A",
    limit=1,
    download=True,
)

first_observation = live_observations[0]
first_product = live_star.products[0]

{
    "available_instruments": live_star.available_instruments(),
    "available_file_types": live_star.available_file_types(),
    "date_range": tuple(date.isoformat() for date in live_star.observation_date_range()),
    "selected_product": first_product,
    "first_observation": first_observation,
    "loaded_observations": len(live_star.observations),
    "downloaded_files": [
        {"path": str(path), "size_mb": round(path.stat().st_size / 1_000_000, 2)}
        for path in sorted((download_dir / "cache" / "downloads").glob("*.fits"))
    ],
}

2026-05-07 18:19:36,129 - WARNING - File .dacerc not found. You are requesting data in public mode. To change this behaviour, create a .dacerc file in your home directory and fill it with your API key. More infos on https://dace.unige.ch
2026-05-07 18:19:39,903 - INFO - Downloading file on location : notebook_scratch/real_dace_download/cache/downloads/r.ESPRE.2019-12-30T00:37:39.686_S1D_A.fits


 Download : 14 MB

2026-05-07 18:19:43,424 - INFO - File downloaded on location : notebook_scratch/real_dace_download/cache/downloads/r.ESPRE.2019-12-30T00:37:39.686_S1D_A.fits


 Download : 15 MB
Download done


{'available_instruments': ['ESPRESSO19'],
 'available_file_types': ['S1D_A'],
 'date_range': ('2019-12-30T00:37:39.686000', '2019-12-30T00:37:39.686000'),
 'selected_product': {'product_id': 563387,
  'spectrum_id': '75079',
  'instrument_name': 'ESPRESSO19',
  'file_rootname': '2019-12-29/r.ESPRE.2019-12-30T00:37:39.686_S1D_A.fits',
  'drs_id': '4',
  'version_major': 3,
  'version_minor': 3,
  'version_patch': 10,
  'rv_extraction_method': 'CCF',
  'file_ext': 'S1D_A'},
 'loaded_observations': 1,
 'downloaded_files': [{'path': 'notebook_scratch/real_dace_download/cache/downloads/r.ESPRE.2019-12-30T00:37:39.686_S1D_A.fits',
   'size_mb': 16.08}]}